# Scaling Monosemanticity mini-replication on Qwen3.5-0.8B

This notebook is a small, local-scale reproduction of the workflow from Anthropic's **Scaling Monosemanticity: Extracting Interpretable Features from Claude 3 Sonnet**. It uses `Qwen/Qwen3.5-0.8B` instead of Claude 3 Sonnet and trains a small sparse autoencoder (SAE) from scratch on middle-layer residual-stream activations.

The notebook saves artifacts under `artifacts/qwen3_5_0_8b_sae/` and skips optional vLLM-based feature labeling when no OpenAI-compatible local endpoint is running.

## Dependencies

Recommended setup from the repository root:

```bash
uv venv
source .venv/bin/activate
uv pip install "transformer-lens>=3" torch transformers datasets pandas numpy plotly tqdm openai pyarrow ipykernel jupyter
python -m ipykernel install --user --name llm-xai-qwen-sae --display-name "llm-xai-qwen-sae"
```

`TransformerBridge` is the intended path for this notebook. If the import check below fails, upgrade to `transformer-lens>=3`.

In [ ]:
# Optional one-shot install from inside the notebook.
# Uncomment when running in a fresh environment.
# %pip install "transformer-lens>=3" torch transformers datasets pandas numpy plotly tqdm openai pyarrow


## Setup

In [ ]:
from __future__ import annotations

import dataclasses
import gc
import html
import importlib
import json
import math
import os
import random
import time
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import plotly.express as px
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

try:
    import transformer_lens
    from transformer_lens.model_bridge import TransformerBridge
except Exception as exc:
    raise RuntimeError(
        "This notebook requires TransformerLens 3 with TransformerBridge. "
        "Install it with: uv pip install 'transformer-lens>=3'"
    ) from exc

print("torch", torch.__version__)
print("transformer_lens", getattr(transformer_lens, "__version__", "unknown"))


In [ ]:
@dataclasses.dataclass
class ExperimentConfig:
    model_id: str = "Qwen/Qwen3.5-0.8B"
    dataset_id: str = "NeelNanda/c4-code-20k"
    dataset_split: str = "train"
    text_column: str | None = None
    artifact_dir: str = "artifacts/qwen3_5_0_8b_sae"
    seed: int = 123
    device: str = "auto"
    dtype: str = "auto"
    smoke_test: bool = True
    num_texts: int = 256
    seq_len: int = 128
    activation_batch_size: int = 4
    train_batch_size: int = 2048
    expansion_factor: int = 8
    num_steps: int = 500
    lr: float = 2e-4
    l1_coeff: float = 5e-3
    checkpoint_every: int = 100
    max_activation_vectors: int = 32768
    activation_key: str | None = None
    num_features_to_explain: int = 8
    top_k_per_feature: int = 20
    vllm_base_url: str = "http://localhost:8000/v1"
    vllm_model: str = "qwen-labeler"
    run_auto_interpretation: bool = True
    run_specificity_scoring: bool = False


CFG = ExperimentConfig()

# Smoke mode is intentionally safe for first execution. Set CFG.smoke_test = False
# after the pipeline works locally.
if CFG.smoke_test:
    CFG.num_texts = 32
    CFG.seq_len = 128
    CFG.activation_batch_size = 2
    CFG.train_batch_size = 512
    CFG.expansion_factor = 2
    CFG.num_steps = 10
    CFG.checkpoint_every = 5
    CFG.max_activation_vectors = 4096
    CFG.num_features_to_explain = 3
    CFG.top_k_per_feature = 8

def choose_device(device: str) -> str:
    if device != "auto":
        return device
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

device = choose_device(CFG.device)
random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)

artifact_dir = Path(CFG.artifact_dir)
checkpoint_dir = artifact_dir / "checkpoints"
example_dir = artifact_dir / "feature_examples"
for path in (artifact_dir, checkpoint_dir, example_dir):
    path.mkdir(parents=True, exist_ok=True)

with open(artifact_dir / "config.json", "w") as f:
    json.dump(dataclasses.asdict(CFG) | {"resolved_device": device}, f, indent=2)

print("device:", device)
print("artifact_dir:", artifact_dir.resolve())
print("HF_TOKEN present:", bool(os.environ.get("HF_TOKEN")))


## Load Qwen through TransformerBridge

This notebook deliberately uses `TransformerBridge.boot_transformers`, not the legacy `HookedTransformer.from_pretrained` API. The bridge preserves raw HuggingFace numerics by default, which is what we want for this Qwen run.

In [ ]:
bridge = TransformerBridge.boot_transformers(CFG.model_id, device=device)
print(type(bridge))

hf_model = getattr(bridge, "model", None) or getattr(bridge, "hf_model", None)
hf_config = getattr(hf_model, "config", None) if hf_model is not None else None
n_layers = getattr(hf_config, "num_hidden_layers", None) or getattr(hf_config, "n_layer", None)
d_model = getattr(hf_config, "hidden_size", None) or getattr(hf_config, "d_model", None)
middle_layer = int(n_layers // 2) if n_layers is not None else None

print("n_layers:", n_layers)
print("d_model:", d_model)
print("middle_layer:", middle_layer)
if d_model is None:
    raise RuntimeError("Could not infer d_model from the bridged model config.")


## Inspect cache keys and select the middle residual stream

In [ ]:
def cache_items(cache: Any) -> list[tuple[Any, Any]]:
    if hasattr(cache, "items"):
        return list(cache.items())
    if hasattr(cache, "cache_dict"):
        return list(cache.cache_dict.items())
    if isinstance(cache, dict):
        return list(cache.items())
    raise TypeError(f"Unsupported cache type: {type(cache)}")

def tensor_cache_items(cache: Any) -> list[tuple[str, torch.Tensor]]:
    rows = []
    for key, value in cache_items(cache):
        if torch.is_tensor(value):
            rows.append((str(key), value))
    return rows

def get_cache_value(cache: Any, key: str) -> torch.Tensor:
    if hasattr(cache, "__getitem__"):
        try:
            return cache[key]
        except Exception:
            pass
    for candidate_key, value in cache_items(cache):
        if str(candidate_key) == key:
            return value
    raise KeyError(key)

def run_bridge_with_cache(inputs: str | list[str]):
    return bridge.run_with_cache(inputs)

logits, sample_cache = run_bridge_with_cache("The Golden Gate Bridge is in San Francisco.")
tensor_keys = tensor_cache_items(sample_cache)
print(f"Cached tensor keys: {len(tensor_keys)}")
for key, value in tensor_keys[:80]:
    print(f"{key:80s} shape={tuple(value.shape)} dtype={value.dtype}")

def infer_activation_key(cache: Any, layer_idx: int | None, d_model: int) -> str:
    if CFG.activation_key is not None:
        return CFG.activation_key

    items = tensor_cache_items(cache)
    layer_terms = [] if layer_idx is None else [
        f".{layer_idx}.", f"_{layer_idx}_", f"/{layer_idx}/", f"layers.{layer_idx}",
        f"blocks.{layer_idx}", f"layer_{layer_idx}", f".{layer_idx}", f"{layer_idx}."
    ]
    resid_terms = ["resid_post", "residual", "hidden_state", "hidden_states", "resid"]

    candidates = []
    for key, value in items:
        key_l = key.lower()
        if value.ndim < 2 or value.shape[-1] != d_model:
            continue
        score = 0
        if any(term in key_l for term in resid_terms):
            score += 10
        if layer_terms and any(term in key_l for term in layer_terms):
            score += 5
        if "post" in key_l or "output" in key_l:
            score += 2
        if "attn" in key_l or "mlp" in key_l:
            score -= 2
        candidates.append((score, key, tuple(value.shape)))

    candidates = sorted(candidates, reverse=True)
    print("Activation key candidates:")
    for row in candidates[:20]:
        print(row)
    if not candidates or candidates[0][0] <= 0:
        raise RuntimeError(
            "Could not infer a residual-stream activation key. Set CFG.activation_key "
            "to one of the printed cache keys and rerun from this cell."
        )
    return candidates[0][1]

activation_key = infer_activation_key(sample_cache, middle_layer, int(d_model))
print("Selected activation key:", activation_key)

cache_meta = {
    "model_id": CFG.model_id,
    "n_layers": n_layers,
    "middle_layer": middle_layer,
    "d_model": d_model,
    "activation_key": activation_key,
    "first_tensor_keys": [{"key": key, "shape": list(value.shape), "dtype": str(value.dtype)} for key, value in tensor_keys[:200]],
}
with open(artifact_dir / "activation_cache_meta.json", "w") as f:
    json.dump(cache_meta, f, indent=2)


## Load a small text dataset

In [ ]:
def load_texts() -> list[str]:
    try:
        from datasets import load_dataset
        ds = load_dataset(CFG.dataset_id, split=CFG.dataset_split)
        columns = list(ds.column_names)
        text_column = CFG.text_column or ("text" if "text" in columns else columns[0])
        texts = [str(x) for x in ds.select(range(min(CFG.num_texts, len(ds))))[text_column]]
        print(f"Loaded {len(texts)} texts from {CFG.dataset_id}[{CFG.dataset_split}], column={text_column!r}")
        return texts
    except Exception as exc:
        print("Dataset load failed; using a tiny built-in fallback corpus.")
        print(type(exc).__name__, exc)
        base = [
            "The Golden Gate Bridge connects San Francisco to Marin County.",
            "A Python function can raise an exception when an input has the wrong type.",
            "Neuroscience studies neurons, synapses, memory, perception, and cognition.",
            "The Eiffel Tower, the Tower of Pisa, and the Sistine Chapel are tourist attractions.",
            "A SQL query can join tables, filter rows, and aggregate values.",
            "A train, a ferry, a tunnel, and a bridge are parts of transit infrastructure.",
            "In machine learning, sparse autoencoders can decompose activations into features.",
            "Security vulnerabilities include buffer overflows, injection bugs, and unsafe deserialization.",
        ]
        return (base * math.ceil(CFG.num_texts / len(base)))[:CFG.num_texts]

texts = load_texts()
texts[:3]


## Collect and normalize middle-layer activations

Anthropic normalizes activations so their average squared L2 norm equals the residual stream dimension `D`. We apply the same scalar normalization before SAE training.

In [ ]:
def get_tokenizer():
    for obj in (bridge, getattr(bridge, "model", None), getattr(bridge, "hf_model", None)):
        tok = getattr(obj, "tokenizer", None)
        if tok is not None:
            return tok
    from transformers import AutoTokenizer
    return AutoTokenizer.from_pretrained(CFG.model_id, trust_remote_code=True)

tokenizer = get_tokenizer()

def ensure_batch_pos_dmodel(x: torch.Tensor) -> torch.Tensor:
    if x.ndim == 2:
        x = x.unsqueeze(0)
    if x.ndim != 3:
        raise ValueError(f"Expected activation tensor with 2 or 3 dims, got {tuple(x.shape)}")
    if x.shape[-1] != d_model and x.shape[0] == d_model:
        x = x.transpose(0, -1)
    return x.detach().float().cpu()

def decode_tokens_for_text(text: str, seq_len: int) -> list[str]:
    encoded = tokenizer(
        text,
        truncation=True,
        max_length=seq_len,
        add_special_tokens=True,
        return_tensors=None,
    )
    ids = encoded.get("input_ids", [])
    if ids and isinstance(ids[0], list):
        ids = ids[0]
    return [tokenizer.decode([tok_id], skip_special_tokens=False) for tok_id in ids]

def chunked(xs: list[str], n: int) -> Iterable[list[str]]:
    for i in range(0, len(xs), n):
        yield xs[i:i+n]

activation_blocks = []
metadata_rows = []

for text_start, batch_texts in enumerate(tqdm(list(chunked(texts, CFG.activation_batch_size)), desc="Collecting activations")):
    batch_index0 = text_start * CFG.activation_batch_size
    try:
        _, cache = run_bridge_with_cache(batch_texts if len(batch_texts) > 1 else batch_texts[0])
        acts = ensure_batch_pos_dmodel(get_cache_value(cache, activation_key))
        if len(batch_texts) == 1 and acts.shape[0] != 1:
            acts = acts[:1]
    except Exception as exc:
        print("Batched bridge call failed; falling back to per-text calls.")
        print(type(exc).__name__, exc)
        per_text = []
        for text in batch_texts:
            _, cache = run_bridge_with_cache(text)
            per_text.append(ensure_batch_pos_dmodel(get_cache_value(cache, activation_key))[0])
        max_len = min(max(x.shape[0] for x in per_text), CFG.seq_len)
        padded = []
        for x in per_text:
            x = x[:max_len]
            if x.shape[0] < max_len:
                pad = torch.zeros(max_len - x.shape[0], x.shape[-1], dtype=x.dtype)
                x = torch.cat([x, pad], dim=0)
            padded.append(x)
        acts = torch.stack(padded, dim=0)

    acts = acts[:, :CFG.seq_len, :]
    for local_idx, text in enumerate(batch_texts):
        text_idx = batch_index0 + local_idx
        token_strings = decode_tokens_for_text(text, CFG.seq_len)
        pos_count = acts.shape[1]
        if len(token_strings) < pos_count:
            token_strings = token_strings + [""] * (pos_count - len(token_strings))
        for pos in range(pos_count):
            metadata_rows.append({
                "row_id": len(metadata_rows),
                "text_idx": text_idx,
                "token_pos": pos,
                "token": token_strings[pos] if pos < len(token_strings) else "",
                "text_preview": text[:240],
            })
    activation_blocks.append(acts.reshape(-1, acts.shape[-1]))
    if sum(block.shape[0] for block in activation_blocks) >= CFG.max_activation_vectors:
        break

raw_acts = torch.cat(activation_blocks, dim=0)[:CFG.max_activation_vectors]
metadata_df = pd.DataFrame(metadata_rows).iloc[:len(raw_acts)].reset_index(drop=True)
metadata_df["row_id"] = np.arange(len(metadata_df))

mean_sq_norm = raw_acts.pow(2).sum(dim=-1).mean().item()
activation_scale = math.sqrt(float(d_model) / max(mean_sq_norm, 1e-12))
acts_norm = raw_acts * activation_scale

print("raw_acts:", tuple(raw_acts.shape))
print("mean_sq_norm before:", mean_sq_norm)
print("activation_scale:", activation_scale)
print("mean_sq_norm after:", acts_norm.pow(2).sum(dim=-1).mean().item())
metadata_df.head()


## Define and train the sparse autoencoder

In [ ]:
class SparseAutoencoder(nn.Module):
    def __init__(self, d_in: int, n_features: int):
        super().__init__()
        self.d_in = d_in
        self.n_features = n_features
        self.b_dec = nn.Parameter(torch.zeros(d_in))
        self.encoder = nn.Linear(d_in, n_features)
        self.W_dec = nn.Parameter(torch.empty(n_features, d_in))
        nn.init.kaiming_uniform_(self.encoder.weight, a=math.sqrt(5))
        nn.init.zeros_(self.encoder.bias)
        nn.init.normal_(self.W_dec, std=1.0 / math.sqrt(d_in))
        self.normalize_decoder_weights()

    @torch.no_grad()
    def normalize_decoder_weights(self):
        self.W_dec.div_(self.W_dec.norm(dim=1, keepdim=True).clamp_min(1e-8))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return F.relu(self.encoder(x - self.b_dec))

    def forward(self, x: torch.Tensor):
        hidden = self.encode(x)
        recon = self.b_dec + hidden @ self.W_dec
        decoder_norms = self.W_dec.norm(dim=1)
        feature_acts = hidden * decoder_norms
        return recon, hidden, feature_acts


n_features = int(d_model) * CFG.expansion_factor
sae = SparseAutoencoder(int(d_model), n_features).to(device)
optimizer = torch.optim.AdamW(sae.parameters(), lr=CFG.lr, betas=(0.9, 0.99), weight_decay=0.0)
train_acts = acts_norm.to(device)

def sae_loss(x: torch.Tensor):
    recon, hidden, feature_acts = sae(x)
    mse = F.mse_loss(recon, x)
    l1 = feature_acts.sum(dim=-1).mean()
    loss = mse + CFG.l1_coeff * l1
    l0 = (hidden > 0).float().sum(dim=-1).mean()
    return loss, {"mse": mse.detach(), "l1": l1.detach(), "mean_l0": l0.detach()}

metrics = []
start = time.time()
for step in tqdm(range(1, CFG.num_steps + 1), desc="Training SAE"):
    batch_idx = torch.randint(0, train_acts.shape[0], (min(CFG.train_batch_size, train_acts.shape[0]),), device=device)
    batch = train_acts[batch_idx]
    optimizer.zero_grad(set_to_none=True)
    loss, parts = sae_loss(batch)
    loss.backward()
    optimizer.step()
    sae.normalize_decoder_weights()

    if step == 1 or step % max(1, CFG.checkpoint_every // 2) == 0 or step == CFG.num_steps:
        metrics.append({
            "step": step,
            "loss": float(loss.detach().cpu()),
            "mse": float(parts["mse"].cpu()),
            "l1": float(parts["l1"].cpu()),
            "mean_l0": float(parts["mean_l0"].cpu()),
            "elapsed_s": time.time() - start,
        })
    if step % CFG.checkpoint_every == 0 or step == CFG.num_steps:
        torch.save({"cfg": dataclasses.asdict(CFG), "step": step, "state_dict": sae.state_dict()}, checkpoint_dir / f"sae_step_{step}.pt")

torch.save({"cfg": dataclasses.asdict(CFG), "state_dict": sae.state_dict(), "activation_scale": activation_scale}, artifact_dir / "sae_final.pt")
metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(artifact_dir / "training_metrics.csv", index=False)
metrics_df


In [ ]:
if not metrics_df.empty:
    display(px.line(metrics_df, x="step", y=["loss", "mse", "l1", "mean_l0"], title="SAE training metrics"))


## Evaluate density, dead features, and reconstruction

In [ ]:
@torch.no_grad()
def compute_feature_stats(x_cpu: torch.Tensor, chunk_size: int = 2048):
    counts = torch.zeros(n_features, dtype=torch.float64)
    max_vals = torch.full((n_features,), -float("inf"), dtype=torch.float64)
    mse_sum = 0.0
    var_sum = 0.0
    n_rows = 0
    l0_sum = 0.0
    sae.eval()
    for start_idx in tqdm(range(0, len(x_cpu), chunk_size), desc="Feature stats"):
        x = x_cpu[start_idx:start_idx+chunk_size].to(device)
        recon, hidden, feature_acts = sae(x)
        active = hidden > 0
        counts += active.sum(dim=0).double().cpu()
        max_vals = torch.maximum(max_vals, feature_acts.max(dim=0).values.double().cpu())
        mse_sum += F.mse_loss(recon, x, reduction="sum").item()
        var_sum += ((x - x.mean(dim=0, keepdim=True)) ** 2).sum().item()
        l0_sum += active.float().sum(dim=-1).sum().item()
        n_rows += x.shape[0]
    freqs = counts / max(n_rows, 1)
    return {
        "freqs": freqs,
        "max_vals": max_vals,
        "mse_per_element": mse_sum / max(n_rows * int(d_model), 1),
        "explained_variance": 1.0 - (mse_sum / max(var_sum, 1e-12)),
        "mean_l0": l0_sum / max(n_rows, 1),
    }

stats = compute_feature_stats(acts_norm)
freq_df = pd.DataFrame({
    "feature_id": np.arange(n_features),
    "frequency": stats["freqs"].numpy(),
    "max_activation": stats["max_vals"].numpy(),
})
freq_df["is_dead"] = freq_df["frequency"] == 0
freq_df.to_csv(artifact_dir / "feature_frequencies.csv", index=False)

summary = {
    "mse_per_element": stats["mse_per_element"],
    "explained_variance": stats["explained_variance"],
    "mean_l0": stats["mean_l0"],
    "dead_feature_fraction": float(freq_df["is_dead"].mean()),
}
with open(artifact_dir / "evaluation_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

summary, freq_df.sort_values("max_activation", ascending=False).head()


In [ ]:
plot_df = freq_df.copy()
plot_df["log10_frequency"] = np.log10(plot_df["frequency"] + 1e-8)
display(px.histogram(plot_df, x="log10_frequency", nbins=80, title="Feature log10 frequency"))


## Find top-activating contexts

In [ ]:
alive = freq_df.query("frequency > 0").copy()
selected_feature_ids = (
    alive.sort_values("max_activation", ascending=False)["feature_id"]
    .head(CFG.num_features_to_explain)
    .astype(int)
    .tolist()
)
selected_feature_ids


In [ ]:
@torch.no_grad()
def selected_feature_activations(x_cpu: torch.Tensor, feature_ids: list[int], chunk_size: int = 2048) -> torch.Tensor:
    cols = []
    feature_ids_t = torch.tensor(feature_ids, device=device, dtype=torch.long)
    sae.eval()
    for start_idx in range(0, len(x_cpu), chunk_size):
        x = x_cpu[start_idx:start_idx+chunk_size].to(device)
        _, _, feature_acts = sae(x)
        cols.append(feature_acts.index_select(1, feature_ids_t).cpu())
    return torch.cat(cols, dim=0)

selected_acts = selected_feature_activations(acts_norm, selected_feature_ids)
top_rows = []
for col_idx, feature_id in enumerate(selected_feature_ids):
    vals = selected_acts[:, col_idx]
    k = min(CFG.top_k_per_feature, len(vals))
    top_vals, top_idx = torch.topk(vals, k=k)
    for rank, (value, row_idx) in enumerate(zip(top_vals.tolist(), top_idx.tolist()), start=1):
        meta = metadata_df.iloc[int(row_idx)].to_dict()
        top_rows.append({
            "feature_id": feature_id,
            "rank": rank,
            "activation": float(value),
            **meta,
        })

top_activations_df = pd.DataFrame(top_rows)
try:
    top_activations_df.to_parquet(artifact_dir / "top_activations.parquet", index=False)
except Exception as exc:
    print("Could not write parquet; writing CSV fallback.")
    print(type(exc).__name__, exc)
    top_activations_df.to_csv(artifact_dir / "top_activations.csv", index=False)

top_activations_df.head(20)


In [ ]:
def activation_to_color(value: float, max_value: float) -> str:
    if max_value <= 0:
        return "background-color: transparent"
    alpha = min(max(value / max_value, 0.0), 1.0)
    return f"background-color: rgba(255, 140, 0, {0.12 + 0.78 * alpha:.3f})"

def write_feature_html(feature_id: int):
    examples = top_activations_df[top_activations_df["feature_id"] == feature_id].head(CFG.top_k_per_feature)
    col_idx = selected_feature_ids.index(feature_id)
    max_value = float(examples["activation"].max()) if len(examples) else 0.0
    parts = ["<html><body>", f"<h1>Feature {feature_id}</h1>"]
    for _, example in examples.iterrows():
        text_idx = int(example["text_idx"])
        same_text = metadata_df[metadata_df["text_idx"] == text_idx].sort_values("token_pos")
        row_ids = same_text["row_id"].astype(int).to_numpy()
        vals = selected_acts[row_ids, col_idx].numpy()
        parts.append(f"<h2>rank {int(example['rank'])}, activation {example['activation']:.4f}</h2>")
        parts.append("<p style='font-family: ui-monospace, SFMono-Regular, Menlo, monospace; line-height: 1.8'>")
        for token, value in zip(same_text["token"].tolist(), vals.tolist()):
            safe_token = html.escape(token).replace("\n", "<br>")
            style = activation_to_color(float(value), max_value)
            parts.append(f"<span title='{value:.4f}' style='{style}'>{safe_token}</span>")
        parts.append("</p>")
    parts.append("</body></html>")
    path = example_dir / f"feature_{feature_id}.html"
    path.write_text("\n".join(parts), encoding="utf-8")
    return path

html_paths = [write_feature_html(fid) for fid in selected_feature_ids]
html_paths


## vLLM feature interpretation

Start an OpenAI-compatible local server in a separate terminal when you want automatic labels:

```bash
pip install vllm openai
vllm serve Qwen/Qwen3.5-0.8B \
  --host 0.0.0.0 \
  --port 8000 \
  --served-model-name qwen-labeler
```

If the base model is not reliable enough at JSON feature labels, use an instruct labeler instead:

```bash
vllm serve Qwen/Qwen2.5-1.5B-Instruct \
  --host 0.0.0.0 \
  --port 8000 \
  --served-model-name qwen-labeler
```

The labeler model does not have to be the same model whose activations the SAE decomposes.

In [ ]:
def vllm_is_available(base_url: str) -> bool:
    try:
        import urllib.request
        with urllib.request.urlopen(base_url.rstrip("/") + "/models", timeout=2) as response:
            return 200 <= response.status < 300
    except Exception:
        return False

def build_label_prompt(feature_id: int, rows: pd.DataFrame) -> str:
    examples = []
    for _, row in rows.head(CFG.top_k_per_feature).iterrows():
        examples.append({
            "rank": int(row["rank"]),
            "activation": round(float(row["activation"]), 4),
            "token": str(row["token"]),
            "context": str(row["text_preview"]),
        })
    return (
        "You are labeling a sparse autoencoder feature from a language model. "
        "Given top activating token contexts, infer the concise concept represented by the feature. "
        "Return only valid JSON with keys: feature_id, label, description, evidence. "
        "Do not include markdown.\n\n"
        f"feature_id: {feature_id}\n"
        f"top_activating_examples: {json.dumps(examples, ensure_ascii=False)}"
    )

def parse_json_object(text: str) -> dict[str, Any]:
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        text = text.removeprefix("json").strip()
    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end >= start:
        text = text[start:end+1]
    return json.loads(text)

labels_path = artifact_dir / "feature_labels.jsonl"
label_records = []

if CFG.run_auto_interpretation and vllm_is_available(CFG.vllm_base_url):
    from openai import OpenAI
    client = OpenAI(base_url=CFG.vllm_base_url, api_key="EMPTY")
    with open(labels_path, "w", encoding="utf-8") as f:
        for feature_id in tqdm(selected_feature_ids, desc="Labeling features"):
            rows = top_activations_df[top_activations_df["feature_id"] == feature_id]
            prompt = build_label_prompt(feature_id, rows)
            response = client.chat.completions.create(
                model=CFG.vllm_model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=300,
            )
            content = response.choices[0].message.content
            try:
                record = parse_json_object(content)
            except Exception:
                record = {"feature_id": feature_id, "label": None, "description": None, "evidence": [], "raw_response": content}
            label_records.append(record)
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
else:
    print(f"Skipping auto-interpretation. No vLLM endpoint at {CFG.vllm_base_url} or CFG.run_auto_interpretation=False.")
    labels_path.write_text("", encoding="utf-8")

label_records[:3]


## Optional specificity scoring

The paper uses a 0-3 rubric for how well a proposed feature description matches activating text:

- 0: irrelevant throughout the context.
- 1: related to the context, but not near the highlighted text or only vaguely related.
- 2: loosely related to the highlighted text or related nearby context.
- 3: cleanly identifies the activating text.

In [ ]:
def build_specificity_prompt(label_record: dict[str, Any], rows: pd.DataFrame) -> str:
    examples = []
    for _, row in rows.head(CFG.top_k_per_feature).iterrows():
        examples.append({
            "rank": int(row["rank"]),
            "activation": round(float(row["activation"]), 4),
            "highlighted_token": str(row["token"]),
            "context": str(row["text_preview"]),
        })
    return (
        "Score how well a sparse-autoencoder feature description matches each activating example. "
        "Use rubric scores 0, 1, 2, or 3. Return only valid JSON with keys feature_id and scores. "
        "Each score item must contain rank, score, and short_reason.\n\n"
        f"feature_description: {json.dumps(label_record, ensure_ascii=False)}\n"
        f"examples: {json.dumps(examples, ensure_ascii=False)}"
    )

specificity_path = artifact_dir / "feature_specificity_scores.jsonl"
specificity_records = []

if CFG.run_specificity_scoring and label_records and vllm_is_available(CFG.vllm_base_url):
    from openai import OpenAI
    client = OpenAI(base_url=CFG.vllm_base_url, api_key="EMPTY")
    with open(specificity_path, "w", encoding="utf-8") as f:
        for record in tqdm(label_records, desc="Specificity scoring"):
            feature_id = int(record["feature_id"])
            rows = top_activations_df[top_activations_df["feature_id"] == feature_id]
            prompt = build_specificity_prompt(record, rows)
            response = client.chat.completions.create(
                model=CFG.vllm_model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=500,
            )
            content = response.choices[0].message.content
            try:
                score_record = parse_json_object(content)
            except Exception:
                score_record = {"feature_id": feature_id, "scores": [], "raw_response": content}
            specificity_records.append(score_record)
            f.write(json.dumps(score_record, ensure_ascii=False) + "\n")
else:
    print("Skipping specificity scoring. Enable CFG.run_specificity_scoring and run vLLM first.")
    specificity_path.write_text("", encoding="utf-8")

specificity_records[:2]


## Artifact checklist

The cells above write the following files when run:

- `config.json`
- `activation_cache_meta.json`
- `sae_final.pt`
- `checkpoints/sae_step_*.pt`
- `training_metrics.csv`
- `feature_frequencies.csv`
- `top_activations.parquet` or `top_activations.csv` fallback
- `feature_labels.jsonl`
- `feature_specificity_scores.jsonl`
- `feature_examples/*.html`

In [ ]:
for path in sorted(artifact_dir.rglob("*")):
    if path.is_file():
        print(path, path.stat().st_size, "bytes")
